# JPMC HR Iceberg Hands-on Lab — Notebook 1: Setup and Synthetic Data

## Overview
This lab demonstrates Snowflake's capabilities on **managed Apache Iceberg tables** using a synthetic HR dataset at production scale (300K employees, ~4.8M total rows).

### Architecture
```
┌──────────────────────┐         ┌──────────────────────────────────┐
│   Databricks         │         │   Snowflake                      │
│   (Gold Layer ETL)   │         │                                  │
│                      │  Iceberg│   ┌──────────────────────────┐   │
│   Writes Parquet +   │────────>│   │  Managed Iceberg Tables  │   │
│   Iceberg metadata   │  format │   │  (GOLD schema)           │   │
│   to S3              │         │   └──────────────────────────┘   │
└──────────────────────┘         │            │                     │
                                 │   Governance, Time Travel,       │
                                 │   Masking, Row Access Policies   │
                                 └──────────────────────────────────┘
```

### What This Notebook Creates
| Object | Detail |
|--------|--------|
| Database | `HOL_HR_ICEBERG_DB` |
| Schemas | `GOLD`, `GOVERNANCE` |
| Roles | `HOL_ADMIN`, `HOL_HR_ANALYST`, `HOL_EMPLOYEE`, `HOL_MANAGER`, `HOL_SENIOR_MANAGER` |
| Warehouses | `HOL_WH_XS` (X-Small), `HOL_WH_S` (Small), `HOL_WH_M` (Medium) |
| Tables | 5 managed Iceberg tables with ~4.8M rows of synthetic HR data |

### Prerequisites
- ACCOUNTADMIN or SYSADMIN role on the Snowflake account
- Ability to create databases, roles, and warehouses

## Reference: Externally Managed Iceberg Tables

> **Note:** In this lab we use **Snowflake-managed** Iceberg tables (data stored and managed by Snowflake) because we don't have access to an external S3 bucket or catalog in this demo environment. The sections below show the DDL you would use in a **production POC** where Databricks writes Iceberg tables to S3 and Snowflake reads them.

### When to Use Which

| Approach | Use When |
|----------|----------|
| **Snowflake-managed** (`CATALOG = 'SNOWFLAKE'`) | Snowflake owns the data lifecycle. Full DML support (INSERT, UPDATE, DELETE, MERGE). |
| **Externally managed — Object Store** | Databricks writes Iceberg to S3. Snowflake reads metadata files directly. You manage refresh. |
| **Externally managed — Unity Catalog** | Databricks Unity Catalog registers the tables. Snowflake connects via REST API with OAuth. |

In [ ]:
/* ============================================================
   REFERENCE: EXTERNAL VOLUME
   Uncomment and customize when connecting to your S3 bucket.
   Run DESC EXTERNAL VOLUME to get the IAM user ARN for the
   trust policy on your AWS account.
   ============================================================ */

-- CREATE OR REPLACE EXTERNAL VOLUME hr_iceberg_ext_vol
--   STORAGE_LOCATIONS = (
--     (
--       NAME                 = 'hr-gold-s3'
--       STORAGE_BASE_URL     = 's3://your-bucket/hr-gold-layer/'
--       STORAGE_PROVIDER     = 'S3'
--       STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::123456789012:role/snowflake-iceberg-role'
--       STORAGE_AWS_EXTERNAL_ID = 'your-external-id'
--     )
--   );

-- DESC EXTERNAL VOLUME hr_iceberg_ext_vol;

In [ ]:
/* ============================================================
   REFERENCE: CATALOG INTEGRATIONS
   Choose ONE based on your setup.
   ============================================================ */

-- ==========================================================
-- Option A: Object Store (RECOMMENDED for this POC)
-- Databricks writes Iceberg tables directly to S3.
-- Snowflake reads the metadata JSON files from S3.
-- You manage the metadata file path manually or via Tasks.
-- ==========================================================

-- CREATE OR REPLACE CATALOG INTEGRATION hr_object_store_cat_int
--   CATALOG_SOURCE = OBJECT_STORE
--   TABLE_FORMAT   = ICEBERG
--   ENABLED        = TRUE;


-- ==========================================================
-- Option B: Databricks Unity Catalog (REST)
-- Use when Unity Catalog manages table registration.
-- Snowflake connects via the Iceberg REST endpoint with OAuth.
-- ==========================================================

-- CREATE OR REPLACE CATALOG INTEGRATION hr_unity_cat_int
--   CATALOG_SOURCE    = ICEBERG_REST
--   TABLE_FORMAT      = ICEBERG
--   CATALOG_URI       = 'https://your-workspace.cloud.databricks.com/api/2.1/unity-catalog/iceberg'
--   CATALOG_WAREHOUSE = 'hr_gold_catalog'
--   REST_CONFIG = (
--     CATALOG_API_TYPE = DATABRICKS
--   )
--   REST_AUTHENTICATION = (
--     TYPE                = OAUTH
--     OAUTH_TOKEN_URI     = 'https://your-workspace.cloud.databricks.com/oidc/v1/token'
--     OAUTH_CLIENT_ID     = 'your-client-id'
--     OAUTH_CLIENT_SECRET = 'your-client-secret'
--     OAUTH_ALLOWED_SCOPES = ('all-apis')
--   )
--   ENABLED = TRUE;

In [ ]:
/* ============================================================
   REFERENCE: EXTERNALLY MANAGED ICEBERG TABLE CREATION
   ============================================================ */

-- === Object Store approach (primary for this POC) ===
-- CREATE ICEBERG TABLE GOLD.EMPLOYEES
--   EXTERNAL_VOLUME    = 'hr_iceberg_ext_vol'
--   CATALOG            = 'hr_object_store_cat_int'
--   METADATA_FILE_PATH = 'employees/metadata/00001-abc123.metadata.json';
--
-- Refresh after Databricks writes new data (pass new metadata path):
-- ALTER ICEBERG TABLE GOLD.EMPLOYEES REFRESH
--   'employees/metadata/00002-def456.metadata.json';
--
-- Automate refresh with a Snowflake Task:
-- CREATE TASK refresh_employees_task
--   WAREHOUSE = HOL_WH_XS
--   SCHEDULE  = 'USING CRON 0 */2 * * * America/New_York'
-- AS
--   ALTER ICEBERG TABLE GOLD.EMPLOYEES REFRESH;


-- === Unity Catalog approach (alternative) ===
-- CREATE ICEBERG TABLE GOLD.EMPLOYEES
--   EXTERNAL_VOLUME    = 'hr_iceberg_ext_vol'
--   CATALOG            = 'hr_unity_cat_int'
--   CATALOG_TABLE_NAME = 'hr_gold_db.employees';
--
-- ALTER ICEBERG TABLE GOLD.EMPLOYEES REFRESH;

---
## Live Lab Setup
Everything below runs on the current Snowflake account using **managed Iceberg tables**.

In [ ]:
-- Step 1: Database and Schemas
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE DATABASE HOL_HR_ICEBERG_DB;
CREATE SCHEMA HOL_HR_ICEBERG_DB.GOLD;
CREATE SCHEMA HOL_HR_ICEBERG_DB.GOVERNANCE;

In [ ]:
-- Step 2: Roles and Hierarchy
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS HOL_ADMIN;
CREATE ROLE IF NOT EXISTS HOL_HR_ANALYST;
CREATE ROLE IF NOT EXISTS HOL_SENIOR_MANAGER;
CREATE ROLE IF NOT EXISTS HOL_MANAGER;
CREATE ROLE IF NOT EXISTS HOL_EMPLOYEE;

-- Hierarchy: ACCOUNTADMIN -> HOL_ADMIN -> HOL_HR_ANALYST
--            HOL_ADMIN -> HOL_SENIOR_MANAGER -> HOL_MANAGER -> HOL_EMPLOYEE
GRANT ROLE HOL_ADMIN          TO ROLE ACCOUNTADMIN;
GRANT ROLE HOL_HR_ANALYST     TO ROLE HOL_ADMIN;
GRANT ROLE HOL_SENIOR_MANAGER TO ROLE HOL_ADMIN;
GRANT ROLE HOL_MANAGER        TO ROLE HOL_SENIOR_MANAGER;
GRANT ROLE HOL_EMPLOYEE       TO ROLE HOL_MANAGER;

-- Database and schema grants
GRANT USAGE ON DATABASE HOL_HR_ICEBERG_DB        TO ROLE HOL_ADMIN;
GRANT USAGE ON SCHEMA   HOL_HR_ICEBERG_DB.GOLD   TO ROLE HOL_ADMIN;
GRANT USAGE ON SCHEMA   HOL_HR_ICEBERG_DB.GOVERNANCE TO ROLE HOL_ADMIN;

GRANT ALL PRIVILEGES ON DATABASE HOL_HR_ICEBERG_DB TO ROLE HOL_ADMIN;
GRANT ALL PRIVILEGES ON SCHEMA   HOL_HR_ICEBERG_DB.GOLD       TO ROLE HOL_ADMIN;
GRANT ALL PRIVILEGES ON SCHEMA   HOL_HR_ICEBERG_DB.GOVERNANCE TO ROLE HOL_ADMIN;

-- Read access for analyst and employee roles (inherited via hierarchy)
GRANT USAGE ON DATABASE HOL_HR_ICEBERG_DB        TO ROLE HOL_HR_ANALYST;
GRANT USAGE ON SCHEMA   HOL_HR_ICEBERG_DB.GOLD   TO ROLE HOL_HR_ANALYST;
GRANT USAGE ON SCHEMA   HOL_HR_ICEBERG_DB.GOVERNANCE TO ROLE HOL_HR_ANALYST;
GRANT SELECT ON ALL TABLES IN SCHEMA HOL_HR_ICEBERG_DB.GOLD       TO ROLE HOL_HR_ANALYST;
GRANT SELECT ON ALL TABLES IN SCHEMA HOL_HR_ICEBERG_DB.GOVERNANCE TO ROLE HOL_HR_ANALYST;

GRANT USAGE ON DATABASE HOL_HR_ICEBERG_DB        TO ROLE HOL_SENIOR_MANAGER;
GRANT USAGE ON SCHEMA   HOL_HR_ICEBERG_DB.GOLD   TO ROLE HOL_SENIOR_MANAGER;
GRANT USAGE ON SCHEMA   HOL_HR_ICEBERG_DB.GOVERNANCE TO ROLE HOL_SENIOR_MANAGER;

-- Future grants so new tables are automatically accessible
GRANT SELECT ON FUTURE TABLES IN SCHEMA HOL_HR_ICEBERG_DB.GOLD       TO ROLE HOL_HR_ANALYST;
GRANT SELECT ON FUTURE TABLES IN SCHEMA HOL_HR_ICEBERG_DB.GOVERNANCE TO ROLE HOL_HR_ANALYST;
GRANT SELECT ON FUTURE TABLES IN SCHEMA HOL_HR_ICEBERG_DB.GOLD       TO ROLE HOL_SENIOR_MANAGER;
GRANT SELECT ON FUTURE TABLES IN SCHEMA HOL_HR_ICEBERG_DB.GOVERNANCE TO ROLE HOL_SENIOR_MANAGER;

-- Grant roles to current user for testing
-- Grant all lab roles to the current user (dynamic, works for any user)
BEGIN
    LET my_user STRING := CURRENT_USER();
    EXECUTE IMMEDIATE 'GRANT ROLE HOL_ADMIN TO USER ' || :my_user;
    EXECUTE IMMEDIATE 'GRANT ROLE HOL_HR_ANALYST TO USER ' || :my_user;
    EXECUTE IMMEDIATE 'GRANT ROLE HOL_SENIOR_MANAGER TO USER ' || :my_user;
    EXECUTE IMMEDIATE 'GRANT ROLE HOL_MANAGER TO USER ' || :my_user;
    EXECUTE IMMEDIATE 'GRANT ROLE HOL_EMPLOYEE TO USER ' || :my_user;
END;

In [ ]:
-- Step 3: Warehouses for sizing demo
USE ROLE ACCOUNTADMIN;

CREATE WAREHOUSE IF NOT EXISTS HOL_WH_XS
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
CREATE WAREHOUSE IF NOT EXISTS HOL_WH_S
  WAREHOUSE_SIZE = 'SMALL'  AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
CREATE WAREHOUSE IF NOT EXISTS HOL_WH_M
  WAREHOUSE_SIZE = 'MEDIUM' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;

GRANT USAGE, OPERATE ON WAREHOUSE HOL_WH_XS TO ROLE HOL_ADMIN;
GRANT USAGE, OPERATE ON WAREHOUSE HOL_WH_S  TO ROLE HOL_ADMIN;
GRANT USAGE, OPERATE ON WAREHOUSE HOL_WH_M  TO ROLE HOL_ADMIN;
GRANT USAGE ON WAREHOUSE HOL_WH_XS TO ROLE HOL_HR_ANALYST;
GRANT USAGE ON WAREHOUSE HOL_WH_XS TO ROLE HOL_SENIOR_MANAGER;
GRANT USAGE ON WAREHOUSE HOL_WH_S  TO ROLE HOL_HR_ANALYST;
GRANT USAGE ON WAREHOUSE HOL_WH_M  TO ROLE HOL_HR_ANALYST;

USE ROLE HOL_ADMIN;
USE WAREHOUSE HOL_WH_XS;

### Step 4: Create Managed Iceberg Tables and Load Synthetic Data

We create 5 tables as **Snowflake-managed Iceberg** (`CATALOG = 'SNOWFLAKE'`, `BASE_LOCATION`).

| Table | Target Rows |
|-------|------------|
| DEPARTMENTS | ~200 |
| EMPLOYEES | ~300,000 |
| COMPENSATION | ~900,000 |
| PERFORMANCE_REVIEWS | ~600,000 |
| ATTENDANCE | ~3,000,000 |

In [ ]:
-- 4a: DEPARTMENTS (~200 rows)
USE ROLE HOL_ADMIN;
USE SCHEMA HOL_HR_ICEBERG_DB.GOLD;

CREATE OR REPLACE ICEBERG TABLE DEPARTMENTS (
    department_id   INT,
    department_name STRING,
    division        STRING,
    cost_center     STRING,
    location_city   STRING
)
  CATALOG = 'SNOWFLAKE';

INSERT INTO DEPARTMENTS
WITH division_data AS (
    SELECT column1 AS division, column2 AS dept_name, column3 AS city
    FROM VALUES
        ('CIB', 'Investment Banking', 'New York'),
        ('CIB', 'Markets - Equities', 'New York'),
        ('CIB', 'Markets - Fixed Income', 'London'),
        ('CIB', 'Markets - Commodities', 'Houston'),
        ('CIB', 'Securities Services', 'New York'),
        ('CIB', 'Commercial Banking', 'Chicago'),
        ('CIB', 'Treasury Services', 'New York'),
        ('CIB', 'Wholesale Payments', 'New York'),
        ('CIB', 'Research', 'London'),
        ('CIB', 'Credit Risk - CIB', 'New York'),
        ('CCB', 'Retail Banking', 'Columbus'),
        ('CCB', 'Card Services', 'Wilmington'),
        ('CCB', 'Auto Finance', 'Chicago'),
        ('CCB', 'Home Lending', 'Columbus'),
        ('CCB', 'Business Banking', 'Columbus'),
        ('CCB', 'Digital Banking', 'San Francisco'),
        ('CCB', 'Branch Operations', 'Columbus'),
        ('CCB', 'Customer Service', 'Phoenix'),
        ('CCB', 'Credit Risk - CCB', 'Columbus'),
        ('CCB', 'Marketing - CCB', 'New York'),
        ('AWM', 'Private Banking', 'New York'),
        ('AWM', 'Asset Management', 'New York'),
        ('AWM', 'Wealth Management', 'San Francisco'),
        ('AWM', 'Fund Services', 'Boston'),
        ('AWM', 'Institutional Sales', 'London'),
        ('AWM', 'Portfolio Management', 'New York'),
        ('AWM', 'Client Advisory', 'New York'),
        ('AWM', 'Alternative Investments', 'New York'),
        ('AWM', 'Retirement Services', 'Columbus'),
        ('AWM', 'Trust Services', 'New York'),
        ('Corporate', 'Human Resources', 'Columbus'),
        ('Corporate', 'Finance', 'New York'),
        ('Corporate', 'Legal', 'New York'),
        ('Corporate', 'Compliance', 'New York'),
        ('Corporate', 'Technology', 'Jersey City'),
        ('Corporate', 'Internal Audit', 'New York'),
        ('Corporate', 'Corporate Strategy', 'New York'),
        ('Corporate', 'Real Estate', 'Columbus'),
        ('Corporate', 'Procurement', 'Chicago'),
        ('Corporate', 'Communications', 'New York')
)
SELECT
    ROW_NUMBER() OVER (ORDER BY division, dept_name) AS department_id,
    dept_name,
    division,
    'CC-' || LPAD(ROW_NUMBER() OVER (ORDER BY division, dept_name)::STRING, 5, '0') AS cost_center,
    city
FROM division_data;

-- Duplicate to reach ~200 departments with regional variants
INSERT INTO DEPARTMENTS
SELECT
    d.department_id + 40 * r.region_offset AS department_id,
    d.department_name || ' - ' || r.region_name AS department_name,
    d.division,
    'CC-' || LPAD((d.department_id + 40 * r.region_offset)::STRING, 5, '0') AS cost_center,
    r.city
FROM DEPARTMENTS d
CROSS JOIN (
    SELECT column1 AS region_offset, column2 AS region_name, column3 AS city FROM VALUES
        (1, 'EMEA', 'London'),
        (2, 'APAC', 'Singapore'),
        (3, 'India', 'Mumbai'),
        (4, 'LATAM', 'Sao Paulo')
) r;

SELECT 'DEPARTMENTS loaded', COUNT(*) AS row_count FROM DEPARTMENTS;

In [ ]:
-- 4b: EMPLOYEES (300K rows, built in 3 waves)
CREATE OR REPLACE ICEBERG TABLE EMPLOYEES (
    employee_id       INT,
    first_name        STRING,
    last_name         STRING,
    email             STRING,
    ssn               STRING,
    phone             STRING,
    department_id     INT,
    manager_id        INT,
    senior_manager_id INT,
    job_title         STRING,
    job_level         INT,
    hire_date         DATE,
    termination_date  DATE,
    employment_status STRING,
    location_city     STRING,
    location_country  STRING
)
  CATALOG = 'SNOWFLAKE';

-- Wave 1: Senior Managers (15,000 rows, job_level 6-8)
INSERT INTO EMPLOYEES
WITH a AS (
  SELECT
    ARRAY_CONSTRUCT('James','Mary','Robert','Patricia','John','Jennifer','Michael','Linda',
      'David','Elizabeth','William','Barbara','Richard','Susan','Joseph','Jessica',
      'Thomas','Sarah','Christopher','Karen','Charles','Lisa','Daniel','Nancy',
      'Matthew','Betty','Anthony','Margaret','Mark','Sandra','Donald','Ashley',
      'Steven','Kimberly','Andrew','Emily','Paul','Donna','Joshua','Michelle',
      'Kenneth','Carol','Kevin','Amanda','Brian','Dorothy','George','Melissa',
      'Timothy','Deborah') AS fns,
    ARRAY_CONSTRUCT('Smith','Johnson','Williams','Brown','Jones','Garcia','Miller','Davis',
      'Rodriguez','Martinez','Hernandez','Lopez','Gonzalez','Wilson','Anderson',
      'Thomas','Taylor','Moore','Jackson','Martin','Lee','Perez','Thompson',
      'White','Harris','Sanchez','Clark','Ramirez','Lewis','Robinson','Walker',
      'Young','Allen','King','Wright','Scott','Torres','Nguyen','Hill','Flores',
      'Green','Adams','Nelson','Baker','Hall','Rivera','Campbell','Mitchell',
      'Carter','Roberts') AS lns,
    ARRAY_CONSTRUCT('New York','London','Singapore','Mumbai','Columbus','Chicago',
      'San Francisco','Houston','Jersey City','Wilmington','Phoenix','Boston',
      'Sao Paulo','Hong Kong','Tokyo') AS cities,
    ARRAY_CONSTRUCT('USA','UK','Singapore','India','USA','USA','USA','USA',
      'USA','USA','USA','USA','Brazil','China','Japan') AS countries
)
SELECT
    seq4()+1,
    a.fns[UNIFORM(0,49,RANDOM())]::STRING,
    a.lns[UNIFORM(0,49,RANDOM())]::STRING,
    LOWER(a.fns[UNIFORM(0,49,RANDOM())]::STRING)||'.'||LOWER(a.lns[UNIFORM(0,49,RANDOM())]::STRING)||(seq4()+1)::STRING||'@jpmc.com',
    LPAD(UNIFORM(100,999,RANDOM())::STRING,3,'0')||'-'||LPAD(UNIFORM(10,99,RANDOM())::STRING,2,'0')||'-'||LPAD(UNIFORM(1000,9999,RANDOM())::STRING,4,'0'),
    '+1-'||UNIFORM(200,999,RANDOM())::STRING||'-'||LPAD(UNIFORM(1000000,9999999,RANDOM())::STRING,7,'0'),
    UNIFORM(1,200,RANDOM()), NULL, NULL,
    ARRAY_CONSTRUCT('Managing Director','Executive Director','Senior Vice President','Global Head','Regional Head')[UNIFORM(0,4,RANDOM())]::STRING,
    UNIFORM(6,8,RANDOM()),
    DATEADD(day,-UNIFORM(1825,7300,RANDOM()),CURRENT_DATE()),
    NULL, 'Active',
    a.cities[UNIFORM(0,14,RANDOM())]::STRING,
    a.countries[UNIFORM(0,14,RANDOM())]::STRING
FROM TABLE(GENERATOR(ROWCOUNT=>15000)), a;

SELECT 'Senior Managers loaded', COUNT(*) AS row_count FROM EMPLOYEES WHERE job_level >= 6;


In [ ]:
-- Wave 2: Managers (45,000 rows, job_level 4-5)
-- Each manager is assigned a random senior_manager_id from Wave 1
INSERT INTO EMPLOYEES
WITH a AS (
  SELECT
    ARRAY_CONSTRUCT('James','Mary','Robert','Patricia','John','Jennifer','Michael','Linda',
      'David','Elizabeth','William','Barbara','Richard','Susan','Joseph','Jessica',
      'Thomas','Sarah','Christopher','Karen','Charles','Lisa','Daniel','Nancy',
      'Matthew','Betty','Anthony','Margaret','Mark','Sandra','Donald','Ashley',
      'Steven','Kimberly','Andrew','Emily','Paul','Donna','Joshua','Michelle',
      'Kenneth','Carol','Kevin','Amanda','Brian','Dorothy','George','Melissa',
      'Timothy','Deborah') AS fns,
    ARRAY_CONSTRUCT('Smith','Johnson','Williams','Brown','Jones','Garcia','Miller','Davis',
      'Rodriguez','Martinez','Hernandez','Lopez','Gonzalez','Wilson','Anderson',
      'Thomas','Taylor','Moore','Jackson','Martin','Lee','Perez','Thompson',
      'White','Harris','Sanchez','Clark','Ramirez','Lewis','Robinson','Walker',
      'Young','Allen','King','Wright','Scott','Torres','Nguyen','Hill','Flores',
      'Green','Adams','Nelson','Baker','Hall','Rivera','Campbell','Mitchell',
      'Carter','Roberts') AS lns,
    ARRAY_CONSTRUCT('New York','London','Singapore','Mumbai','Columbus','Chicago',
      'San Francisco','Houston','Jersey City','Wilmington','Phoenix','Boston',
      'Sao Paulo','Hong Kong','Tokyo') AS cities,
    ARRAY_CONSTRUCT('USA','UK','Singapore','India','USA','USA','USA','USA',
      'USA','USA','USA','USA','Brazil','China','Japan') AS countries
)
SELECT
    15000+seq4()+1,
    a.fns[UNIFORM(0,49,RANDOM())]::STRING,
    a.lns[UNIFORM(0,49,RANDOM())]::STRING,
    LOWER(a.fns[UNIFORM(0,49,RANDOM())]::STRING)||'.'||LOWER(a.lns[UNIFORM(0,49,RANDOM())]::STRING)||(15000+seq4()+1)::STRING||'@jpmc.com',
    LPAD(UNIFORM(100,999,RANDOM())::STRING,3,'0')||'-'||LPAD(UNIFORM(10,99,RANDOM())::STRING,2,'0')||'-'||LPAD(UNIFORM(1000,9999,RANDOM())::STRING,4,'0'),
    '+1-'||UNIFORM(200,999,RANDOM())::STRING||'-'||LPAD(UNIFORM(1000000,9999999,RANDOM())::STRING,7,'0'),
    UNIFORM(1,200,RANDOM()), NULL, UNIFORM(1,15000,RANDOM()),
    ARRAY_CONSTRUCT('Vice President','Director','Senior Manager','Team Lead','Department Manager')[UNIFORM(0,4,RANDOM())]::STRING,
    UNIFORM(4,5,RANDOM()),
    DATEADD(day,-UNIFORM(730,5475,RANDOM()),CURRENT_DATE()),
    NULL, 'Active',
    a.cities[UNIFORM(0,14,RANDOM())]::STRING,
    a.countries[UNIFORM(0,14,RANDOM())]::STRING
FROM TABLE(GENERATOR(ROWCOUNT=>45000)), a;

SELECT 'Managers loaded', COUNT(*) AS row_count FROM EMPLOYEES WHERE job_level BETWEEN 4 AND 5;


In [ ]:
-- Wave 3: Individual Contributors (240,000 rows, job_level 1-3)
-- Each IC is assigned a manager (from Wave 2) and a senior manager (from Wave 1)
INSERT INTO EMPLOYEES
WITH a AS (
  SELECT
    ARRAY_CONSTRUCT('James','Mary','Robert','Patricia','John','Jennifer','Michael','Linda',
      'David','Elizabeth','William','Barbara','Richard','Susan','Joseph','Jessica',
      'Thomas','Sarah','Christopher','Karen','Charles','Lisa','Daniel','Nancy',
      'Matthew','Betty','Anthony','Margaret','Mark','Sandra','Donald','Ashley',
      'Steven','Kimberly','Andrew','Emily','Paul','Donna','Joshua','Michelle',
      'Kenneth','Carol','Kevin','Amanda','Brian','Dorothy','George','Melissa',
      'Timothy','Deborah') AS fns,
    ARRAY_CONSTRUCT('Smith','Johnson','Williams','Brown','Jones','Garcia','Miller','Davis',
      'Rodriguez','Martinez','Hernandez','Lopez','Gonzalez','Wilson','Anderson',
      'Thomas','Taylor','Moore','Jackson','Martin','Lee','Perez','Thompson',
      'White','Harris','Sanchez','Clark','Ramirez','Lewis','Robinson','Walker',
      'Young','Allen','King','Wright','Scott','Torres','Nguyen','Hill','Flores',
      'Green','Adams','Nelson','Baker','Hall','Rivera','Campbell','Mitchell',
      'Carter','Roberts') AS lns,
    ARRAY_CONSTRUCT('New York','London','Singapore','Mumbai','Columbus','Chicago',
      'San Francisco','Houston','Jersey City','Wilmington','Phoenix','Boston',
      'Sao Paulo','Hong Kong','Tokyo') AS cities,
    ARRAY_CONSTRUCT('USA','UK','Singapore','India','USA','USA','USA','USA',
      'USA','USA','USA','USA','Brazil','China','Japan') AS countries
)
SELECT
    60000+seq4()+1,
    a.fns[UNIFORM(0,49,RANDOM())]::STRING,
    a.lns[UNIFORM(0,49,RANDOM())]::STRING,
    LOWER(a.fns[UNIFORM(0,49,RANDOM())]::STRING)||'.'||LOWER(a.lns[UNIFORM(0,49,RANDOM())]::STRING)||(60000+seq4()+1)::STRING||'@jpmc.com',
    LPAD(UNIFORM(100,999,RANDOM())::STRING,3,'0')||'-'||LPAD(UNIFORM(10,99,RANDOM())::STRING,2,'0')||'-'||LPAD(UNIFORM(1000,9999,RANDOM())::STRING,4,'0'),
    '+1-'||UNIFORM(200,999,RANDOM())::STRING||'-'||LPAD(UNIFORM(1000000,9999999,RANDOM())::STRING,7,'0'),
    UNIFORM(1,200,RANDOM()),
    UNIFORM(15001,60000,RANDOM()),
    UNIFORM(1,15000,RANDOM()),
    ARRAY_CONSTRUCT('Analyst','Associate','Senior Associate','Specialist','Senior Analyst',
      'Operations Analyst','Quantitative Analyst','Risk Analyst','Compliance Analyst','Technology Analyst')[UNIFORM(0,9,RANDOM())]::STRING,
    UNIFORM(1,3,RANDOM()),
    DATEADD(day,-UNIFORM(30,3650,RANDOM()),CURRENT_DATE()),
    NULL,
    CASE WHEN UNIFORM(1,100,RANDOM())<=5 THEN 'Terminated' ELSE 'Active' END,
    a.cities[UNIFORM(0,14,RANDOM())]::STRING,
    a.countries[UNIFORM(0,14,RANDOM())]::STRING
FROM TABLE(GENERATOR(ROWCOUNT=>240000)), a;

SELECT 'Total Employees', COUNT(*) AS row_count FROM EMPLOYEES;
SELECT job_level, COUNT(*) AS cnt FROM EMPLOYEES GROUP BY job_level ORDER BY job_level;


In [ ]:
-- 4c: COMPENSATION (~900K rows, 3 records per employee for 2022/2023/2024)
CREATE OR REPLACE ICEBERG TABLE COMPENSATION (
    comp_id        INT,
    employee_id    INT,
    effective_date DATE,
    base_salary    NUMBER(12,2),
    bonus_pct      NUMBER(5,2),
    bonus_amount   NUMBER(12,2),
    stock_units    INT,
    total_comp     NUMBER(12,2),
    currency       STRING,
    pay_grade      STRING
)
  CATALOG = 'SNOWFLAKE';

INSERT INTO COMPENSATION
WITH emp AS (SELECT employee_id, job_level FROM EMPLOYEES),
years AS (SELECT column1 AS yr FROM VALUES (2022),(2023),(2024))
SELECT
    ROW_NUMBER() OVER (ORDER BY e.employee_id, y.yr),
    e.employee_id,
    DATE_FROM_PARTS(y.yr, 1, 1),
    ROUND(CASE
        WHEN e.job_level BETWEEN 1 AND 3 THEN UNIFORM(55000,120000,RANDOM())
        WHEN e.job_level BETWEEN 4 AND 5 THEN UNIFORM(120000,250000,RANDOM())
        ELSE UNIFORM(250000,600000,RANDOM())
    END * (1 + (y.yr - 2022) * 0.04), 2),
    ROUND(UNIFORM(5,35,RANDOM()) + e.job_level * 3, 2),
    NULL,
    CASE WHEN e.job_level>=6 THEN UNIFORM(500,5000,RANDOM())
         WHEN e.job_level>=4 THEN UNIFORM(100,1000,RANDOM())
         ELSE UNIFORM(0,200,RANDOM()) END,
    NULL,
    'USD',
    'PG-'||e.job_level::STRING
FROM emp e CROSS JOIN years y;

-- Fill calculated columns
UPDATE COMPENSATION SET
    bonus_amount = ROUND(base_salary * bonus_pct / 100, 2),
    total_comp = ROUND(base_salary + (base_salary * bonus_pct / 100) + (stock_units * 150), 2);

SELECT 'COMPENSATION loaded', COUNT(*) AS row_count FROM COMPENSATION;


In [ ]:
-- 4d: PERFORMANCE_REVIEWS (~600K rows, 2 per employee)
CREATE OR REPLACE ICEBERG TABLE PERFORMANCE_REVIEWS (
    review_id         INT,
    employee_id       INT,
    reviewer_id       INT,
    review_period     STRING,
    overall_rating    NUMBER(2,1),
    goal_rating       NUMBER(2,1),
    competency_rating NUMBER(2,1),
    comments          STRING,
    review_date       DATE
)
  CATALOG = 'SNOWFLAKE';

INSERT INTO PERFORMANCE_REVIEWS
WITH emp AS (SELECT employee_id, manager_id FROM EMPLOYEES),
periods AS (SELECT column1 AS period, column2::DATE AS rev_date FROM VALUES ('H1-2024','2024-07-15'),('H2-2023','2024-01-15')),
ct AS (SELECT ARRAY_CONSTRUCT(
    'Consistently exceeds expectations. Strong leadership.',
    'Meets expectations. Good team contributor.',
    'Shows improvement. Needs development in key areas.',
    'Excellent performance across all metrics.',
    'Solid performer. Ready for next-level responsibilities.',
    'Demonstrates strong analytical skills.',
    'Needs to improve communication and collaboration.',
    'Outstanding client engagement and delivery.',
    'Reliable performer with consistent output.',
    'Shows initiative and drives innovation.') AS comments)
SELECT
    ROW_NUMBER() OVER (ORDER BY e.employee_id, p.period),
    e.employee_id,
    COALESCE(e.manager_id, e.employee_id),
    p.period,
    ROUND(UNIFORM(10,50,RANDOM())/10.0,1),
    ROUND(UNIFORM(10,50,RANDOM())/10.0,1),
    ROUND(UNIFORM(10,50,RANDOM())/10.0,1),
    ct.comments[UNIFORM(0,9,RANDOM())]::STRING,
    p.rev_date
FROM emp e CROSS JOIN periods p CROSS JOIN ct;

SELECT 'PERFORMANCE_REVIEWS loaded', COUNT(*) AS row_count FROM PERFORMANCE_REVIEWS;


In [ ]:
-- 4e: ATTENDANCE (~3M rows, 10 per employee)
CREATE OR REPLACE ICEBERG TABLE ATTENDANCE (
    attendance_id   INT,
    employee_id     INT,
    attendance_date DATE,
    status          STRING,
    hours_worked    NUMBER(4,1),
    check_in_time   TIME,
    check_out_time  TIME
)
  CATALOG = 'SNOWFLAKE';

INSERT INTO ATTENDANCE
WITH emp AS (SELECT employee_id FROM EMPLOYEES),
days AS (SELECT seq4() AS d FROM TABLE(GENERATOR(ROWCOUNT=>10))),
s AS (SELECT ARRAY_CONSTRUCT('Present','Present','Present','Present','Present','Present','WFH','WFH','WFH','PTO','Sick','Holiday') AS statuses)
SELECT
    ROW_NUMBER() OVER (ORDER BY e.employee_id, d.d),
    e.employee_id,
    DATEADD(day, -d.d * UNIFORM(1,3,RANDOM()), CURRENT_DATE()),
    s.statuses[UNIFORM(0,11,RANDOM())]::STRING,
    ROUND(UNIFORM(40,100,RANDOM())/10.0,1),
    TIME_FROM_PARTS(UNIFORM(7,10,RANDOM()),UNIFORM(0,59,RANDOM()),0),
    TIME_FROM_PARTS(UNIFORM(16,20,RANDOM()),UNIFORM(0,59,RANDOM()),0)
FROM emp e CROSS JOIN days d CROSS JOIN s;

SELECT 'ATTENDANCE loaded', COUNT(*) AS row_count FROM ATTENDANCE;


In [ ]:
-- Step 5: Role-to-Employee Mapping for Row Access Policies
USE SCHEMA HOL_HR_ICEBERG_DB.GOVERNANCE;

CREATE OR REPLACE ICEBERG TABLE ROLE_ACCESS_MAPPING (
    role_name     STRING,
    employee_id   INT,
    department_id INT
)
  CATALOG = 'SNOWFLAKE';

-- Pick a specific employee for each role demo
-- HOL_EMPLOYEE: employee_id = 100000 (an individual contributor)
-- HOL_MANAGER: the manager of employee 100000
-- HOL_SENIOR_MANAGER: the senior manager of employee 100000
INSERT INTO ROLE_ACCESS_MAPPING
WITH target_emp AS (
    SELECT employee_id, manager_id, senior_manager_id, department_id
    FROM HOL_HR_ICEBERG_DB.GOLD.EMPLOYEES
    WHERE employee_id = 100000
)
SELECT 'HOL_EMPLOYEE', employee_id, department_id FROM target_emp
UNION ALL
SELECT 'HOL_MANAGER', manager_id, department_id FROM target_emp
UNION ALL
SELECT 'HOL_SENIOR_MANAGER', senior_manager_id,
       (SELECT department_id FROM HOL_HR_ICEBERG_DB.GOLD.EMPLOYEES WHERE employee_id = (SELECT senior_manager_id FROM target_emp))
FROM target_emp;

-- Grant SELECT on mapping table to all roles
GRANT SELECT ON TABLE ROLE_ACCESS_MAPPING TO ROLE HOL_HR_ANALYST;
GRANT SELECT ON TABLE ROLE_ACCESS_MAPPING TO ROLE HOL_SENIOR_MANAGER;

SELECT * FROM ROLE_ACCESS_MAPPING;

In [ ]:
-- Step 6: Verification
USE SCHEMA HOL_HR_ICEBERG_DB.GOLD;

-- Row counts
SELECT 'DEPARTMENTS'         AS table_name, COUNT(*) AS row_count FROM DEPARTMENTS
UNION ALL
SELECT 'EMPLOYEES',           COUNT(*) FROM EMPLOYEES
UNION ALL
SELECT 'COMPENSATION',        COUNT(*) FROM COMPENSATION
UNION ALL
SELECT 'PERFORMANCE_REVIEWS', COUNT(*) FROM PERFORMANCE_REVIEWS
UNION ALL
SELECT 'ATTENDANCE',          COUNT(*) FROM ATTENDANCE;

-- Confirm they are Iceberg tables
SHOW ICEBERG TABLES IN SCHEMA HOL_HR_ICEBERG_DB.GOLD;

-- Sample data
SELECT * FROM EMPLOYEES LIMIT 5;
SELECT * FROM COMPENSATION LIMIT 5;

---
## Summary

Setup complete. Objects created:
- **Database**: `HOL_HR_ICEBERG_DB` with schemas `GOLD` and `GOVERNANCE`
- **5 Roles**: HOL_ADMIN, HOL_HR_ANALYST, HOL_EMPLOYEE, HOL_MANAGER, HOL_SENIOR_MANAGER
- **3 Warehouses**: HOL_WH_XS, HOL_WH_S, HOL_WH_M
- **5 Iceberg Tables**: DEPARTMENTS (~200), EMPLOYEES (~300K), COMPENSATION (~900K), PERFORMANCE_REVIEWS (~600K), ATTENDANCE (~3M)
- **Mapping Table**: GOVERNANCE.ROLE_ACCESS_MAPPING

**Next**: Open `02_Features_and_Governance.ipynb` for time travel, warehouse sizing, dynamic masking, and row-level access policy demos.